# Laboratorio di Sismologia — picking P/S e spettro di Brune

**Terremoto del 10 marzo 2014 — California settentrionale.** In questa esercitazione identificherai gli arrivi P/S sulle tre componenti, selezionerai una finestra, fitterai lo spettro con il modello di Brune e confronterai le magnitudo ottenute da più stazioni.

**Modulo 0 — picking:** scegli soltanto la **stazione**: sono mostrate Z, N ed E con la stessa scala dei tempi. Sposta la **linea rossa** con il cursore sopra il grafico e premi **Segna P** oppure **Segna S**. Per posizionarla con precisione, fai clic sulla manopola dello slider e usa le frecce **←/→** della tastiera: ogni pressione sposta di un campione (0,005 o 0,01 s secondo la stazione). Sono disponibili anche due pulsanti **◀ 1 campione / 1 campione ▶** e un campo numerico. Se il grafico interattivo è disponibile, un clic sulla traccia sposta il cursore rosso e i tasti P/S assegnano il pick corrispondente. Puoi fare zoom sul grafico; il pulsante **Home** riporta alla vista iniziale senza spostare le tracce.

**Modulo 1 — spettro:** seleziona indipendentemente stazione, componente e fase. È necessario aver assegnato **entrambi i pick P e S** alla stazione per eseguire il fit e registrare una misura.

**Modulo finale — confronto statistico:** registra più misure, visualizza i gruppi e confronta deviazione standard, errore standard e valore di catalogo.

I pick precedenti negli header SAC **non vengono usati**. Il pacchetto per gli studenti include anche i file SAC con i pick originali rimossi. Apri il notebook in Jupyter e scegli **Cell → Run All**. Il sorgente viene nascosto e si può ripristinare con **Mostra codice**.


In [ ]:
# In JupyterLite ipywidgets può non essere preinstallato.
# Questo blocco lo installa soltanto nel browser, senza modificare Python sul PC.
try:
    import ipywidgets as widgets
except ModuleNotFoundError as exc:
    if exc.name != "ipywidgets":
        raise
    try:
        import piplite
    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "ipywidgets non è installato: fuori da JupyterLite va installato "
            "nell'ambiente Python scelto. Nessuna installazione automatica locale è stata avviata."
        ) from exc
    await piplite.install("ipywidgets")
    import ipywidgets as widgets

from pathlib import Path
from zipfile import ZipFile
from IPython.display import display, clear_output, HTML, Image
from io import BytesIO
import csv
import html
import struct

import numpy as np
import matplotlib.pyplot as plt

# Interazione diretta con Matplotlib solo se ipympl è davvero installato.
# In Jupyter Classic il backend nbagg dentro widgets.Output può non ricevere
# correttamente gli eventi; non lo usiamo come fallback.
PICKER_BACKEND = 'statico'
try:
    import ipympl
    get_ipython().run_line_magic('matplotlib', 'widget')
    PICKER_BACKEND = 'widget'
except Exception:
    # Il plot statico più i widget nativi funzionano senza ipympl.
    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception:
        pass
plt.ioff()
# P e S sono scorciatoie del corso: disattiva i comandi Matplotlib
# normalmente associati a 'p' (pan) e 's' (salva immagine).
plt.rcParams['keymap.pan'] = []
plt.rcParams['keymap.save'] = []
# Le frecce sono riservate al posizionamento fine del cursore del pick.
plt.rcParams['keymap.back'] = []
plt.rcParams['keymap.forward'] = []

def mostra_figura_png(fig):
    """Render statico delle analisi: non crea canvas interattive extra."""
    buf = BytesIO()
    try:
        fig.savefig(buf, format='png', dpi=115, bbox_inches='tight')
        display(Image(data=buf.getvalue()))
    finally:
        plt.close(fig)
        buf.close()

from scipy import signal
from scipy.optimize import least_squares
import ipywidgets as widgets

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.16, "font.size": 10})
# All'interno del notebook classico, CSS nasconde il sorgente ma non gli output.
# L'area resta disponibile per il pulsante «Mostra codice» del pannello.
code_style_output = widgets.Output(layout=widgets.Layout(height='0px', min_height='0px', overflow='hidden'))
display(code_style_output)


In [ ]:
# ---- Funzioni numeriche (non serve modificarle per svolgere l'esercitazione) ----

def sac_val(value):
    return None if (not np.isfinite(value) or np.isclose(value, -12345.0)) else float(value)


def leggi_sac_bytes(blob):
    """Legge i SAC binari classici (header 632 byte; serie float32)."""
    if len(blob) < 636:
        raise ValueError("File SAC troppo corto")
    little = struct.unpack('<i', blob[304:308])[0]
    big = struct.unpack('>i', blob[304:308])[0]
    if little in (6, 7):
        endian = '<'
    elif big in (6, 7):
        endian = '>'
    else:
        raise ValueError("Formato SAC non riconosciuto (nvhdr)")
    ff = struct.unpack(endian+'70f', blob[:280])
    ii = struct.unpack(endian+'40i', blob[280:440])
    n = int(ii[9])
    if n <= 0 or not 0 < ff[0] < 10 or len(blob) < 632+4*n:
        raise ValueError("Header SAC: delta, npts o dimensione non validi")
    y = np.frombuffer(blob, dtype=endian+'f4', offset=632, count=n).astype(np.float64)
    strings = blob[440:632]
    def s(i):
        return strings[i*8:(i+1)*8].decode('ascii',errors='replace').replace('\x00','').strip()
    return dict(
        y=y, dt=float(ff[0]), b=float(ff[5]), p=None, s=None,
        dist_km=sac_val(ff[50]), evdp_km=sac_val(ff[38]), mag=sac_val(ff[39]),
        user3=sac_val(ff[43]), station=s(0), channel=s(20), network=s(21)
    )


def carica_tracce(percorso):
    """Accetta una cartella SAC, anche con sottocartelle, o un file ZIP."""
    p = Path(percorso).expanduser()
    if not p.is_absolute():
        p = Path.cwd() / p
    if p.is_dir():
        entries = ((str(f), f.read_bytes()) for f in sorted(p.rglob('*')) if f.is_file() and f.suffix.lower()=='.sac')
    elif p.is_file() and p.suffix.lower()=='.zip':
        with ZipFile(p) as z:
            content = [(name, z.read(name)) for name in z.namelist() if name.lower().endswith('.sac')]
        entries = iter(content)
    else:
        raise FileNotFoundError(f"Non trovo una cartella SAC o uno ZIP: {p}")

    loaded = {}
    for filename, blob in entries:
        name = Path(filename).name
        if name.lower() == 'a.sac':
            continue  # file duplicato nel dataset fornito
        parts = name.split('.')
        if len(parts) < 4:
            continue
        st, component = parts[-3], parts[-2].upper()
        if component not in ('HNE', 'HNN', 'HNZ'):
            continue
        trace = leggi_sac_bytes(blob)
        if trace['user3'] is None or trace['user3'] <= 0:
            raise ValueError(f"Campo user3 assente/non valido in {name}")
        loaded[(st,component)] = trace
    if not loaded:
        raise ValueError("Nessun SAC riconosciuto: attesi nomi ...STAZIONE.HNE/HNN/HNZ.sac")

    # I valori T0/T2 originari dell'header NON sono usati. p/s sono
    # inizialmente None e saranno impostati soltanto dal picking manuale.
    return loaded


def accelerazione_si(tr):
    return tr['y']/tr['user3']


def brune(f, omega0, fc):
    return omega0/(1+(np.asarray(f)/fc)**2)


def spettro(tr, start, end):
    """FFT continua approssimata (scala dt) dello spostamento, in m*s."""
    i0 = max(0, int(round((start-tr['b'])/tr['dt'])))
    i1 = min(len(tr['y']), int(round((end-tr['b'])/tr['dt'])))
    a = accelerazione_si(tr)[i0:i1]
    if len(a)*tr['dt'] < 3 or len(a) < 100:
        raise ValueError("Scegli almeno 3 s di segnale")
    if not np.all(np.isfinite(a)):
        raise ValueError("Campioni non finiti")
    a = signal.detrend(a, type='linear')
    a = a * signal.windows.tukey(len(a), alpha=.1)
    f = np.fft.rfftfreq(len(a), d=tr['dt'])
    amp = np.abs(np.fft.rfft(a)) * tr['dt']
    amp[1:] /= (2*np.pi*f[1:])**2
    amp[0] = np.nan
    return f, amp, len(a)


def fit_brune(f, amp, fmin, fmax):
    good = np.isfinite(amp) & (amp > 0) & (f >= fmin) & (f <= fmax)
    fq, ap = f[good], amp[good]
    if len(fq) < 12:
        raise ValueError("Meno di 12 frequenze: aumenta la finestra o allarga la banda")
    initial = [np.log10(np.median(ap[:max(3,len(ap)//8)])), np.log10(np.sqrt(fq[0]*fq[-1]))]
    lower = [-35., np.log10(max(fq[0]/3, 1e-4))]
    upper = [35., np.log10(fq[-1]*3)]
    initial = np.clip(initial, np.asarray(lower)+.001, np.asarray(upper)-.001)
    def errors(p):
        return np.log10(brune(fq,10**p[0],10**p[1])) - np.log10(ap)
    result = least_squares(errors, initial, bounds=(lower, upper), loss='soft_l1', f_scale=.15)
    return 10**result.x[0], 10**result.x[1], np.sqrt(np.mean(result.fun**2))


def momento_magnitudo(tr, phase, omega0):
    if tr['dist_km'] is None or tr['evdp_km'] is None:
        raise ValueError("Distanza epicentrale o profondità non disponibili nell'header")
    rho = 2700.0  # kg/m³
    velocity = 3500.0 if phase == 'S' else 6000.0  # m/s
    radiation = 0.63 if phase == 'S' else 0.52
    free_surface = 2.0
    hypo_m = 1000*np.hypot(tr['dist_km'], tr['evdp_km'])
    M0 = (4*np.pi*rho*(velocity**3)*hypo_m*omega0)/(radiation*free_surface)
    Mw = (2/3)*(np.log10(M0)-9.1)
    return M0, Mw


In [ ]:
# ---- INTERFACCIA: dopo aver eseguito questa cella, usa solo i menu e i pulsanti ----
state = {'tracce': {}, 'busy': False, 'modo': 'manuale', 'misure': [],
         'picks': {}, 'pick_fig': None, 'pick_axes': None, 'last_click_time': None,
         'pick_plot_station': None, 'pick_artists': None,
         'pick_default_window': None}

# Caricamento SENZA modificare il codice: basta scegliere la cartella in questo campo.
def percorso_iniziale():
    folder = Path('2014.03.10.05.18.13')
    if folder.is_dir():
        return str(folder)
    if any(Path.cwd().glob('*.sac')):
        return '.'
    archive = Path('2014.03.10.05.18.13.zip')
    return str(archive if archive.is_file() else folder)

path_box = widgets.Text(value=percorso_iniziale(), description='Cartella SAC:',
                        placeholder='Incolla qui un percorso di cartella o .zip',
                        layout=widgets.Layout(width='78%'), style={'description_width':'110px'})
load_button = widgets.Button(description='Carica dati', button_style='primary', icon='folder-open')
load_status = widgets.HTML(value='<i>Carica le registrazioni per iniziare.</i>')

station = widgets.Dropdown(options=[], description='Stazione:',
                           style={'description_width':'95px'}, layout=widgets.Layout(width='245px'))
component = widgets.Dropdown(options=[], description='Componente:',
                             style={'description_width':'100px'}, layout=widgets.Layout(width='245px'))
phase = widgets.ToggleButtons(options=[('Fase S','S'),('Fase P','P')], value='S',
                              description='Fase:', style={'description_width':'55px'},
                              layout=widgets.Layout(width='250px'))

window = widgets.FloatRangeSlider(value=[0., 5.], min=0., max=100., step=.25,
                                  description='Finestra (s):', continuous_update=False,
                                  readout=True, readout_format='.1f',
                                  style={'description_width':'110px'},
                                  layout=widgets.Layout(width='96%'))
window_label = widgets.HTML(value='')
reset_window = widgets.Button(description='Finestra suggerita', icon='refresh')
zoom_window = widgets.Checkbox(value=False, description='Zoom sul segmento', indent=False, layout=widgets.Layout(width='200px'))

fmin = widgets.BoundedFloatText(value=.08, min=.001, max=50., step=.01,
                                description='f min (Hz):', style={'description_width':'110px'})
fmax = widgets.BoundedFloatText(value=5.0, min=.002, max=100., step=.5,
                                description='f max (Hz):', style={'description_width':'110px'})
logomega = widgets.FloatSlider(value=-.5, min=-12., max=4., step=.05, continuous_update=False,
                               readout_format='.2f', description='log₁₀ Ω₀:',
                               style={'description_width':'110px'}, layout=widgets.Layout(width='96%'))
fc = widgets.FloatLogSlider(value=.25, base=10, min=-2.5, max=1.5, step=.025,
                            continuous_update=False, description='fc (Hz):',
                            readout_format='.3g', style={'description_width':'110px'},
                            layout=widgets.Layout(width='96%'))
fit_button = widgets.Button(description='Fit automatico', button_style='info', icon='line-chart')
record_button = widgets.Button(description='Registra misura', button_style='success', icon='plus-circle')
export_button = widgets.Button(description='Esporta CSV', icon='download')
metrics = widgets.HTML(value='')
waveform = widgets.Output()
spectrum_output = widgets.Output()
records = widgets.Output()
notice = widgets.HTML(value='')

# MODULO 0: scelta indipendente della stazione e picking sulle tre componenti.
# La scelta di stazione/componente/fase per lo spettro è solo nel Modulo 1.
pick_station = widgets.Dropdown(options=[], description='Stazione:',
                                style={'description_width': '95px'},
                                layout=widgets.Layout(width='245px'))
pick_graph = widgets.Output(layout=widgets.Layout(width='100%'))
pick_notice = widgets.HTML(value='')
pick_summary = widgets.HTML(value='')
pick_view = widgets.FloatRangeSlider(description='Zoom temporale (s):', min=-15., max=320.,
                                     value=[0., 90.], step=.25, continuous_update=False,
                                     readout_format='.1f',
                                     style={'description_width': '145px'},
                                     layout=widgets.Layout(width='99%'))
# Comandi di zoom nativi: funzionano anche con il backend statico di JupyterLite.
pick_zoom_in_button = widgets.Button(description='Zoom +', icon='search-plus',
                                     tooltip='Dimezza la finestra temporale intorno al cursore',
                                     layout=widgets.Layout(width='125px'))
pick_zoom_out_button = widgets.Button(description='Zoom −', icon='search-minus',
                                      tooltip='Raddoppia la finestra temporale intorno al cursore',
                                      layout=widgets.Layout(width='125px'))
pick_home_button = widgets.Button(description='Vista iniziale', icon='home',
                                  tooltip='Ripristina la finestra iniziale della stazione',
                                  layout=widgets.Layout(width='155px'))

# Cursore nativo ipywidgets: NON dipende dagli eventi del grafico Matplotlib.
# Le frecce sinistra/destra, col cursore focalizzato, avanzano di un campione;
# il passo è aggiornato automaticamente in funzione del campionamento della stazione.
pick_time_slider = widgets.FloatSlider(value=30., min=0., max=90., step=.005,
                                       description='Posizione pick (s):', continuous_update=False,
                                       readout=True, readout_format='.3f',
                                       style={'description_width':'145px'},
                                       layout=widgets.Layout(width='99%'))
pick_time_input = widgets.FloatText(value=30., description='Tempo preciso (s):', step=.001,
                                    style={'description_width': '125px'},
                                    layout=widgets.Layout(width='250px'))
pick_left_button = widgets.Button(description='◀ 1 campione', tooltip='Sposta il cursore a sinistra di un campione', layout=widgets.Layout(width='145px'))
pick_right_button = widgets.Button(description='1 campione ▶', tooltip='Sposta il cursore a destra di un campione', layout=widgets.Layout(width='145px'))
pick_P_button = widgets.Button(description='Segna P', icon='map-pin', button_style='warning')
pick_S_button = widgets.Button(description='Segna S', icon='map-pin', button_style='success')
pick_clear_button = widgets.Button(description='Cancella pick stazione', icon='eraser')
pick_export_button = widgets.Button(description='Esporta pick CSV', icon='download')


def has_student_picks(sta=None):
    name = sta if sta is not None else station.value  # Modulo 1
    entry = state['picks'].get(name, {})
    return entry.get('P') is not None and entry.get('S') is not None


def require_student_picks():
    if not has_student_picks():
        raise ValueError('Nel Modulo 0 devi prima segnare sia P sia S per questa stazione.')


def update_picks_table():
    if not state['tracce']:
        pick_summary.value = ''
        return
    rows = ['<table style="font-size:12px;border-collapse:collapse">',
            '<tr><th style="text-align:left;padding:4px 12px">Stazione</th>'
            '<th style="padding:4px 12px">P (s)</th><th style="padding:4px 12px">S (s)</th>'
            '<th style="padding:4px 12px">Stato</th></tr>']
    for name in sorted({k[0] for k in state['tracce']}):
        picks = state['picks'].get(name,{})
        p,s = picks.get('P'),picks.get('S')
        mark = '✓ Completo' if p is not None and s is not None else 'Da completare'
        rows.append('<tr>' + ''.join(
            f'<td style="padding:3px 12px;border-bottom:1px solid #ddd">{html.escape(str(v))}</td>'
            for v in (name, '—' if p is None else f'{p:.3f}',
                      '—' if s is None else f'{s:.3f}', mark)) + '</tr>')
    pick_summary.value = ''.join(rows)+'</table>'


def picker_time_bounds(sta):
    traces = [tr for (name,ch),tr in state['tracce'].items() if name == sta]
    return (min(tr['b'] for tr in traces),
            max(tr['b']+(len(tr['y'])-1)*tr['dt'] for tr in traces))


def configure_pick_controls():
    """Configura stazione e cursori del modulo 0."""
    if not state['tracce'] or pick_station.value is None:
        return
    sta = pick_station.value
    first, last = picker_time_bounds(sta)
    sample_dt = min(tr['dt'] for (name, _), tr in state['tracce'].items() if name == sta)
    first, last = float(np.floor(first*4)/4), float(np.ceil(last*4)/4)
    state['busy'] = True
    try:
        pick_view.min = min(first, pick_view.min)
        pick_view.max = max(last, pick_view.max)
        pick_view.min = first
        pick_view.max = last
        left, right = max(first, 0.), min(last, 90.)
        if right-left < 8:
            left, right = first, last
        pick_view.value = (left,right)
        pick_view.step = float(min(0.05, sample_dt))  # zoom fine anche a campioni ravvicinati
        state['pick_default_window'] = (float(left),float(right))
        pick_time_slider.min = min(left, pick_time_slider.min)
        pick_time_slider.max = max(right, pick_time_slider.max)
        pick_time_slider.step = float(sample_dt)
        pick_time_slider.min = left
        pick_time_slider.max = right
        time = 0.5*(left+right)
        pick_time_slider.value = float(round(time, 3))
        pick_time_input.value = float(round(time, 3))
        state['last_click_time'] = None
    finally:
        state['busy'] = False
    update_picker_plot(reset_view=True)
    update_picks_table()


def time_view_changed(change=None):
    if state['busy'] or not state['tracce']:
        return
    left,right = pick_view.value
    state['busy'] = True
    try:
        current = min(max(pick_time_slider.value,left),right)
        pick_time_slider.min = min(pick_time_slider.min,left)
        pick_time_slider.max = max(pick_time_slider.max,right)
        pick_time_slider.min = left
        pick_time_slider.max = right
        pick_time_slider.value = float(np.clip(current,left,right))
        pick_time_input.value = float(pick_time_slider.value)
    finally:
        state['busy'] = False
    # Il range scelto da 'Zoom temporale' deve diventare la nuova vista.
    update_picker_plot(reset_view=True)


def change_picker_zoom(factor):
    """Modifica soltanto la finestra temporale del Modulo 0, senza toolbar.

    factor=0.5 per ingrandire, 2 per allargare; il centro segue il cursore
    quando è dentro la finestra e resta nei limiti temporali della stazione.
    """
    if state['busy'] or not state['tracce'] or pick_station.value is None:
        return
    low, high = float(pick_view.min), float(pick_view.max)
    if high <= low:
        return
    left, right = map(float, pick_view.value)
    dt = min(tr['dt'] for (name, _), tr in state['tracce'].items()
             if name == pick_station.value)
    min_width = min(high-low, max(0.10, 4*float(dt)))
    new_width = float(np.clip((right-left)*factor, min_width, high-low))
    # Il cursore determina il centro dello zoom; altrimenti si usa il centro vista.
    current = float(pick_time_input.value)
    center = current if left <= current <= right else (left+right)/2
    new_left = float(np.clip(center-new_width/2, low, high-new_width))
    new_right = new_left + new_width
    if abs(new_left-left) < 1e-9 and abs(new_right-right) < 1e-9:
        return
    # Lo stesso observer già esistente aggiorna slider, grafici e cursore.
    pick_view.value = (new_left, new_right)


def restore_picker_view(_=None):
    """Ripristina il range suggerito per la stazione, senza cancellare i pick."""
    if state['busy'] or not state['tracce']:
        return
    initial = state.get('pick_default_window')
    if initial is not None:
        pick_view.value = initial


def autoscale_picker_visible_y(axes, sta):
    """Scala ogni componente sull'ampiezza del tratto visibile.

    Serve specialmente con i grafici statici: un piccolo arrivo P altrimenti
    rischia di essere schiacciato dall'ampiezza della successiva fase S.
    """
    left, right = axes[-1].get_xlim()
    for ax, ch in zip(axes, ('HNZ', 'HNN', 'HNE')):
        tr = state['tracce'].get((sta, ch))
        if tr is None or tr['dt'] <= 0:
            continue
        start = max(0, int(np.floor((left-tr['b'])/tr['dt'])))
        stop = min(len(tr['y']), int(np.ceil((right-tr['b'])/tr['dt']))+1)
        if stop <= start:
            continue
        visible = np.asarray(accelerazione_si(tr)[start:stop])
        visible = visible[np.isfinite(visible)]
        if not len(visible):
            continue
        vmin, vmax = float(visible.min()), float(visible.max())
        margin = max((vmax-vmin)*0.12, max(abs(vmin),abs(vmax))*0.03, 1e-12)
        ax.set_ylim(vmin-margin, vmax+margin)



def slider_pick_changed(change=None):
    if state['busy'] or not state['tracce']:
        return
    state['busy'] = True
    try:
        pick_time_input.value = float(pick_time_slider.value)
    finally:
        state['busy'] = False
    update_picker_plot()


def typed_pick_changed(change=None):
    if state['busy'] or not state['tracce'] or pick_station.value is None:
        return
    t = float(pick_time_input.value)
    if not np.isfinite(t):
        return
    lo,hi = picker_time_bounds(pick_station.value)
    if not lo <= t <= hi:
        pick_notice.value = '<b style="color:#a33">Tempo digitato fuori dalla registrazione.</b>'
        return
    moved_view = False
    if not pick_view.value[0] <= t <= pick_view.value[1]:
        # Non arrotondare il valore digitato. Riposiziona la vista sul tempo.
        moved_view = True
        left,right=pick_view.value
        width=max(4.,right-left)
        a=max(pick_view.min, min(t-width/2.,pick_view.max-width))
        b=min(pick_view.max,a+width)
        state['busy']=True
        try:
            pick_view.value=(a,b)
            pick_time_slider.min=min(a,pick_time_slider.min)
            pick_time_slider.max=max(b,pick_time_slider.max)
            pick_time_slider.min=a
            pick_time_slider.max=b
        finally:
            state['busy']=False
    state['busy']=True
    try:
        pick_time_slider.value=float(t)
    finally:
        state['busy']=False
    update_picker_plot(reset_view=moved_view)


def move_picker_cursor(direction):
    """Sposta il cursore di un campione nel sistema temporale della stazione.

    Usato dai tasti freccia quando il grafico ha il focus, dai normali tasti
    freccia sullo slider ipywidgets e dai pulsanti di riserva sul pannello.
    """
    if not state['tracce'] or pick_station.value is None:
        return
    sta = pick_station.value
    dt = min(tr['dt'] for (name, _), tr in state['tracce'].items() if name == sta)
    lo, hi = picker_time_bounds(sta)
    t = float(np.clip(round(float(pick_time_input.value) + int(direction)*dt, 6), lo, hi))
    if np.isclose(t, pick_time_input.value, atol=1e-10, rtol=0):
        return
    # L'observer del campo si occupa di aggiornare anche slider e linea rossa.
    pick_time_input.value = t


def ensure_picker_canvas():
    if PICKER_BACKEND != 'widget' or state['pick_fig'] is not None:
        return
    try:
        # Evitiamo constrained_layout: il layout dinamico può cambiare la
        # geometria degli assi memorizzata dalla toolbar Home di Matplotlib.
        fig, axes = plt.subplots(3, 1, figsize=(10.4, 6.6), sharex=True)
        fig.subplots_adjust(left=.12, right=.98, top=.92, bottom=.11, hspace=.13)
        state['pick_fig'], state['pick_axes'] = fig, list(axes)
        state['pick_artists'] = None
        fig.canvas.mpl_connect('button_press_event', on_picker_click)
        fig.canvas.mpl_connect('key_press_event', on_picker_key)
        with pick_graph:
            clear_output(wait=True)
            fig.canvas.toolbar_visible = True
            fig.canvas.header_visible = False
            fig.canvas.resizable = False
            display(fig.canvas)
    except Exception as exc:
        state['pick_fig'], state['pick_axes'] = None, None
        state['pick_artists'] = None
        pick_notice.value = ('<b style="color:#a33">Canvas interattivo non disponibile: </b>'
                             + html.escape(str(exc)) + '. Usa il cursore e Segna P/S.')


def update_picker_plot(reset_view=False):
    """Aggiorna solo le linee mobili e NON ricrea gli assi durante il picking.

    La cronologia della toolbar (inclusa la casetta Home) resta coerente
    perché zoom e pan modificano gli assi originali, che non vengono svuotati
    spostando il cursore o registrando P/S. Solo un cambio di stazione o
    un nuovo caricamento ricostruiscono le forme d'onda e la vista iniziale.
    """
    if not state['tracce'] or pick_station.value is None:
        return
    sta = pick_station.value
    entry = state['picks'].get(sta, {})
    if PICKER_BACKEND == 'widget' and state['pick_fig'] is None:
        ensure_picker_canvas()

    interactive = PICKER_BACKEND == 'widget' and state['pick_fig'] is not None
    if interactive:
        fig, axes = state['pick_fig'], state['pick_axes']
    else:
        fig, axes = plt.subplots(3, 1, figsize=(10.4, 6.6), sharex=True,
                                 constrained_layout=True)

    rebuild = (not interactive or state.get('pick_plot_station') != sta
               or state.get('pick_artists') is None)
    if rebuild:
        artists = {'cursor': [], 'P': [], 'S': []}
        for ax, ch, label in zip(axes, ('HNZ', 'HNN', 'HNE'), ('Z', 'N', 'E')):
            ax.clear()
            tr = state['tracce'].get((sta, ch))
            if tr is not None:
                t = tr['b'] + np.arange(len(tr['y']))*tr['dt']
                y = accelerazione_si(tr)
                stride = max(1, len(y)//16000)
                ax.plot(t[::stride], y[::stride], lw=.7, color='#3b6e88')
            else:
                ax.text(.5, .5, f'{ch} non disponibile', ha='center',
                        va='center', transform=ax.transAxes)
            # Creiamo le linee una sola volta. Aggiornandone poi xdata/visible
            # la toolbar Home continua a riferirsi agli stessi assi.
            p_line = ax.axvline(0, color='#d07b36', ls='--', lw=1.7,
                                label='P', zorder=8, visible=False)
            s_line = ax.axvline(0, color='#118875', ls='--', lw=1.7,
                                label='S', zorder=8, visible=False)
            cursor_line = ax.axvline(0, color='#d62728', ls='-', lw=2.0,
                                     alpha=.95, label='Cursore', zorder=12)
            artists['P'].append(p_line)
            artists['S'].append(s_line)
            artists['cursor'].append(cursor_line)
            ax.set_ylabel(f'{label} ({ch})\n(m/s²)', fontsize=9)
            ax.grid(True, alpha=.16)
        axes[0].set_title(f'Modulo 0 · Stazione {sta} · posiziona il cursore e segna P/S')
        axes[-1].set_xlabel('Tempo relativo al riferimento SAC (s)')
        axes[-1].set_xlim(*pick_view.value)
        if interactive:
            state['pick_artists'] = artists
            state['pick_plot_station'] = sta
            # Ricostruisce la posizione iniziale della toolbar quando cambia
            # stazione, senza lasciare nell'Home i limiti di quella precedente.
            toolbar = getattr(fig.canvas, 'toolbar', None)
            try:
                fig.canvas.draw()  # stabilizza la geometria prima dello snapshot
                if toolbar is not None and hasattr(toolbar, 'update'):
                    toolbar.update()  # azzera la vecchia cronologia
                    if hasattr(toolbar, 'push_current'):
                        toolbar.push_current()  # Home = vista di questa stazione
            except Exception:
                pass  # la modalità slider rimane utilizzabile
    else:
        artists = state['pick_artists']

    for ph in ('P', 'S'):
        picked = entry.get(ph)
        for line in artists[ph]:
            line.set_visible(picked is not None)
            if picked is not None:
                line.set_xdata((picked, picked))
    t_cursor = float(pick_time_input.value)
    for line in artists['cursor']:
        line.set_visible(np.isfinite(t_cursor))
        if np.isfinite(t_cursor):
            line.set_xdata((t_cursor, t_cursor))

    # La legenda deve riflettere i pick effettivamente registrati.
    # Usiamo gli artisti esistenti senza ricreare gli assi, per non alterare
    # lo zoom e la cronologia del pulsante Home di Matplotlib.
    legend_handles = [artists['cursor'][0]]
    legend_labels = ['Cursore']
    for ph in ('P', 'S'):
        picked = entry.get(ph)
        if picked is not None:
            legend_handles.append(artists[ph][0])
            legend_labels.append(f'{ph}: {picked:.3f} s')
    current_legend = axes[0].get_legend()
    if (current_legend is None or
            [item.get_text() for item in current_legend.get_texts()] != legend_labels):
        axes[0].legend(legend_handles, legend_labels, loc='upper right',
                       fontsize=8, ncol=3, framealpha=.9)

    # Lo zoom scelto dal pannello può cambiare esplicitamente la vista,
    # ma il semplice movimento della linea rossa non tocca mai xlim / ylim.
    if reset_view:
        axes[-1].set_xlim(*pick_view.value)
    if reset_view or not interactive:
        autoscale_picker_visible_y(axes, sta)

    if interactive:
        fig.canvas.draw_idle()
    else:
        with pick_graph:
            clear_output(wait=True)
            mostra_figura_png(fig)


def write_student_pick(kind, t=None, origin='manuale'):
    """Salva il tempo esatto del clic; non passa attraverso lo slider."""
    if not state['tracce'] or pick_station.value is None:
        return
    name=pick_station.value
    if t is None:
        t = pick_time_input.value
    t=float(t)
    limits=picker_time_bounds(name)
    if not np.isfinite(t) or not (limits[0] <= t <= limits[1]):
        pick_notice.value='<b style="color:#a33">Tempo fuori dalla registrazione.</b>'
        return
    old=state['picks'].get(name,{'P':None,'S':None})
    other=old['S' if kind=='P' else 'P']
    if other is not None and ((kind=='P' and t>=other) or (kind=='S' and t<=other)):
        pick_notice.value='<b style="color:#a33">Serve P &lt; S. Cambia il punto e riprova.</b>'
        return
    if old.get(kind) is not None and np.isclose(old[kind],t,rtol=0,atol=1e-8):
        return
    entry=dict(old)
    entry[kind]=t
    state['picks'][name]=entry
    for (sta,ch),tr in state['tracce'].items():
        if sta==name:
            tr[kind.lower()]=t
    remove_stale_measurements(name)
    update_picks_table()
    # Il campo numerico è solo un'indicazione: il valore salvato è preciso.
    # Aggiorniamo il cursore senza alterare l'attuale zoom.
    state['busy'] = True
    try:
        pick_time_input.value = t
        if pick_time_slider.min <= t <= pick_time_slider.max:
            pick_time_slider.value = t
    finally:
        state['busy'] = False
    update_picker_plot()
    # Aggiorna la finestra dello spettro soltanto se riguarda la stessa stazione.
    if station.value==name:
        configure_window()
    pick_notice.value=(f'<b>Stazione {html.escape(str(name))}:</b> '
                       f'pick {kind} = {t:.4f} s ({origin}). '
                       + ('Entrambi assegnati: ora puoi selezionare stazione, componente e fase nel Modulo 1.'
                          if has_student_picks(name) else 'Ora assegna anche l’altro pick.'))


def on_picker_click(event):
    """Il clic imposta il cursore rosso; i pulsanti decidono se è P oppure S."""
    if (event.button != 1 or event.inaxes not in (state.get('pick_axes') or [])
            or event.xdata is None):
        return
    toolbar = getattr(event.canvas, 'toolbar', None)
    if toolbar is not None and getattr(toolbar, 'mode', ''):
        return  # il clic appartiene alla funzione Zoom/Pan attiva
    state['last_click_time'] = float(event.xdata)
    pick_time_input.value = float(event.xdata)  # sincronizza anche il cursore


def on_picker_key(event):
    """Frecce = un campione; P/S salvano il tempo del cursore rosso."""
    if not event.key:
        return
    key = event.key.lower()
    if key in ('left', 'right'):
        move_picker_cursor(-1 if key == 'left' else 1)
        return
    if key not in ('p', 's'):
        return
    toolbar = getattr(event.canvas, 'toolbar', None)
    if toolbar is not None and getattr(toolbar, 'mode', ''):
        return
    write_student_pick(key.upper(), t=float(pick_time_input.value), origin='tastiera')


def remove_stale_measurements(sta):
    count_before=len(state['misure'])
    state['misure']=[m for m in state['misure'] if m['stazione']!=sta]
    if len(state['misure'])!=count_before:
        show_records()
        if 'aggiorna_statistiche' in globals():
            aggiorna_statistiche()
        notice.value=(f'<i>Modificati i pick di {html.escape(str(sta))}: '
                      'misure precedenti di questa stazione eliminate, per coerenza.</i>')


def clear_station_picks(_=None):
    if not state['tracce'] or pick_station.value is None:
        return
    sta=pick_station.value
    state['picks'][sta]={'P':None,'S':None}
    for (name,ch),tr in state['tracce'].items():
        if name==sta:
            tr['p']=tr['s']=None
    remove_stale_measurements(sta)
    update_picks_table()
    update_picker_plot()
    if station.value == sta:
        configure_window()
    pick_notice.value=f'<i>Pick della stazione {html.escape(str(sta))} cancellati.</i>'


def export_student_picks(_=None):
    if not state['tracce']:
        return
    target=Path('picks_studenti_Brune.csv')
    with target.open('w',newline='',encoding='utf-8-sig') as fp:
        writer=csv.DictWriter(fp,fieldnames=['stazione','P_s','S_s'])
        writer.writeheader()
        for sta in sorted({k[0] for k in state['tracce']}):
            picks=state['picks'].get(sta,{})
            writer.writerow({'stazione':sta,'P_s':picks.get('P'),'S_s':picks.get('S')})
    pick_notice.value=f'<b>Pick salvati:</b> {html.escape(str(target.resolve()))}'

pick_zoom_in_button.on_click(lambda b: change_picker_zoom(0.5))
pick_zoom_out_button.on_click(lambda b: change_picker_zoom(2.0))
pick_home_button.on_click(restore_picker_view)
pick_view.observe(time_view_changed, names='value')
pick_time_slider.observe(slider_pick_changed, names='value')
pick_time_input.observe(typed_pick_changed, names='value')
pick_station.observe(lambda change: configure_pick_controls() if not state['busy'] else None, names='value')
pick_P_button.on_click(lambda b: write_student_pick('P', origin='cursore / tempo numerico'))
pick_S_button.on_click(lambda b: write_student_pick('S', origin='cursore / tempo numerico'))
pick_left_button.on_click(lambda b: move_picker_cursor(-1))
pick_right_button.on_click(lambda b: move_picker_cursor(1))
pick_clear_button.on_click(clear_station_picks)
pick_export_button.on_click(export_student_picks)


def current_trace():
    return state['tracce'][(station.value, component.value)]


def default_window():
    tr = current_trace()
    t_first = tr['b']
    t_last = tr['b'] + (len(tr['y'])-1)*tr['dt']
    pick = tr['s'] if phase.value=='S' else tr['p']
    if pick is None:
        t0 = max(t_first, t_first+.1*(t_last-t_first))
        t1 = min(t_last, t0+30)
    elif phase.value=='P':
        t0 = max(t_first, pick + .25)
        # Non includere per default la fase S nella finestra P.
        t1 = min(t_last, pick+12, (tr['s']-.5) if tr['s'] is not None else t_last)
    else:
        t0 = max(t_first, pick + .25)
        t1 = min(t_last, pick+45)
    if t1-t0 < 3:
        t0, t1 = max(t_first, t0-1), min(t_last, t0+5)
    return t0, t1


def configure_window(use_default=True):
    if not state['tracce'] or station.value is None or component.value is None:
        return
    tr = current_trace()
    first, last = tr['b'], tr['b']+(len(tr['y'])-1)*tr['dt']
    state['busy'] = True
    try:
        window.min = float(np.floor(first*4)/4)
        window.max = float(np.ceil(last*4)/4)
        if use_default:
            start,end = default_window()
            window.value = (float(round(start*4)/4),float(round(end*4)/4))
    finally:
        state['busy'] = False
    redraw()


def load_click(_=None):
    state['busy'] = True
    try:
        loaded = carica_tracce(path_box.value)
        state['tracce'] = loaded
        state['picks'] = {st: {'P':None,'S':None} for st,c in loaded}
        state['misure'] = []
        state['pick_plot_station'] = None
        state['pick_artists'] = None
        stations = sorted({st for st,c in loaded})
        pick_station.options = stations
        pick_station.value = '1023' if '1023' in stations else stations[0]
        station.options = stations
        station.value = pick_station.value
        available = [c for c in ('HNE','HNN','HNZ') if (station.value,c) in loaded]
        component.options = available
        component.value = 'HNN' if 'HNN' in available else available[0]
        load_status.value = (f'<b style="color:#16803c">✓ Caricate {len(loaded)} tracce '
                             f'di {len(stations)} stazioni.</b> Ora usa i menu qui sotto.')
        state['modo'] = 'manuale'
    except Exception as exc:
        state['tracce'] = {}
        pick_station.options = []
        station.options = []
        component.options = []
        load_status.value = f'<b style="color:#b91c1c">Errore:</b> {html.escape(str(exc))}'
        for out in (waveform, spectrum_output):
            with out:
                clear_output(wait=True)
        metrics.value = ''
        return
    finally:
        state['busy'] = False
    configure_pick_controls()
    configure_window()
    if 'aggiorna_statistiche' in globals():
        aggiorna_statistiche()


def station_changed(change=None):
    if state['busy'] or not state['tracce'] or station.value is None:
        return
    state['busy'] = True
    try:
        options = [c for c in ('HNE','HNN','HNZ') if (station.value,c) in state['tracce']]
        component.options = options
        preference = 'HNZ' if phase.value=='P' else 'HNN'
        component.value = preference if preference in options else options[0]
        state['modo'] = 'manuale'
    finally:
        state['busy'] = False
    configure_window()


def phase_changed(change=None):
    if state['busy'] or not state['tracce']:
        return
    state['busy'] = True
    try:
        pref = 'HNZ' if phase.value=='P' else 'HNN'
        if pref in component.options:
            component.value = pref
        state['modo'] = 'manuale'
    finally:
        state['busy'] = False
    configure_window()


def component_changed(change=None):
    if state['busy'] or not state['tracce']:
        return
    state['modo'] = 'manuale'
    configure_window()


def redraw(change=None):
    """Aggiorna i grafici: forma d'onda in alto, spettro e parametri in basso."""
    if state['busy'] or not state['tracce']:
        return
    tr = current_trace()
    start, end = window.value
    pick = tr['s'] if phase.value == 'S' else tr['p']
    duration = end-start
    resolution = 1 / max(duration, 1e-10)
    pick_label = f'Il tuo pick {phase.value}: {pick:.2f} s' if pick is not None else f'Il tuo pick {phase.value} non ancora assegnato'
    window_label.value = (
        f'<b>Finestra:</b> {start:.2f}–{end:.2f} s '
        f'({duration:.2f} s) &nbsp; · &nbsp; '
        f'<b>Risoluzione Δf:</b> {resolution:.3f} Hz &nbsp; · &nbsp; {pick_label}'
    )

    # 1. SISMOGRAMMA
    with waveform:
        clear_output(wait=True)
        time = tr['b'] + np.arange(len(tr['y']))*tr['dt']
        acc = accelerazione_si(tr)
        stride = max(1, len(time)//12000)
        fig, ax = plt.subplots(figsize=(10.6, 3.1), constrained_layout=True)
        ax.plot(time[::stride], acc[::stride], color='#356a83', lw=.75)
        if tr['p'] is not None:
            ax.axvline(tr['p'], color='#d07b36', ls='--', lw=1.3, label='P')
        if tr['s'] is not None:
            ax.axvline(tr['s'], color='#118875', ls='--', lw=1.3, label='S')
        ax.axvspan(start, end, color='#f0b44b', alpha=.34, label='Finestra')
        if zoom_window.value:
            center = (start+end)/2
            half_span = max((end-start)*1.05, 8.)
            ax.set_xlim(max(time[0],center-half_span), min(time[-1],center+half_span))
        else:
            ax.set_xlim(time[0],time[-1])
        ax.set(title=f'Stazione {station.value} · {component.value} · fase {phase.value}',
               xlabel='Tempo relativo al riferimento SAC (s)',
               ylabel='Accelerazione (m/s²)')
        ax.legend(loc='upper right', fontsize=9, ncol=3, framealpha=.9)
        mostra_figura_png(fig)

    # 2. SPETTRO E CALCOLO DI Mw
    with spectrum_output:
        clear_output(wait=True)
        try:
            require_student_picks()
            if duration < 3:
                raise ValueError('Finestra troppo corta: servono almeno 3 s')
            if not (0 < fmin.value < fmax.value < .5/tr['dt']):
                raise ValueError('La banda deve essere positiva e sotto la frequenza di Nyquist')
            freqs, amplitudes, npts = spettro(tr, start, end)
            good = (freqs > 0) & np.isfinite(amplitudes) & (amplitudes > 0)
            if not np.any(good):
                raise ValueError('Spettro privo di ampiezze utilizzabili')
            fig, ax = plt.subplots(figsize=(10.6, 4.0), constrained_layout=True)
            ax.loglog(freqs[good], amplitudes[good], lw=.85,
                      color='#57718a', alpha=.9, label='Spettro osservato')
            ax.axvspan(fmin.value, fmax.value, color='#6eaeb4', alpha=.18,
                       label='Banda del fit')
            om = 10**logomega.value
            upper_freq = min(freqs[-1], 100.)
            if upper_freq <= max(freqs[1], .001):
                raise ValueError('Frequenze insufficienti per visualizzare il modello')
            fplot = np.geomspace(max(freqs[1], .001), upper_freq, 400)
            ax.loglog(fplot, brune(fplot, om, fc.value), color='#b84354', lw=2.25,
                      label=f'Modello di Brune ({state["modo"]})')
            ax.axhline(om, ls=':', lw=1.2, color='#b84354', label='Plateau Ω₀')
            ax.axvline(fc.value, ls='--', lw=1.2, color='#b84354', label='Corner frequency fc')
            ax.set_xlim(max(freqs[1], .01), min(50., freqs[-1]))
            ax.set(xlabel='Frequenza (Hz)', ylabel='Spettro di spostamento (m·s)')
            ax.legend(loc='best', fontsize=9, ncol=2)
            mostra_figura_png(fig)

            M0, Mw = momento_magnitudo(tr, phase.value, om)
            catmag = f'{tr["mag"]:.2f}' if tr['mag'] is not None else 'n.d.'
            warnings = []
            if fc.value < 2*resolution:
                warnings.append('fc < 2Δf: corner frequency poco risolta')
            if phase.value == 'P' and tr['s'] is not None and end > tr['s']:
                warnings.append('La finestra P contiene anche S')
            if not (fmin.value <= fc.value <= fmax.value):
                warnings.append('fc fuori dalla banda di fit')
            if start >= end:
                warnings.append('Finestra non valida')
            caution = ("<div style='color:#95601b;margin-top:5px'>"+
                       ' · '.join(html.escape(s) for s in warnings)+"</div>") if warnings else ''
            metrics.value = (
                '<div style="padding:12px 14px;border:1px solid #ccd8e0;border-radius:8px">'
                f'<span style="font-size:14px"><b>Ω₀</b> = {om:.3e} m·s &nbsp; · &nbsp; '
                f'<b>fc</b> = {fc.value:.3f} Hz &nbsp; · &nbsp; '
                f'<b>M₀</b> = {M0:.3e} N·m</span><br>'
                f'<span style="font-size:20px"><b>Mw stimata: {Mw:.2f}</b></span>'
                f' &nbsp; · &nbsp; Mw di catalogo: <b>{catmag}</b>'
                '<div style="font-size:12px;color:#64748b">'
                'Stima didattica con correzione geometrica semplificata; '
                'non include attenuazione anelastica ed effetti di sito.'
                '</div>'
                f'{caution}</div>'
            )
            notice.value = ''
        except Exception as exc:
            metrics.value = ''
            notice.value = f'<span style="color:#a33">{html.escape(str(exc))}</span>'


def fit_click(_):
    if not state['tracce']:
        return
    try:
        tr = current_trace()
        require_student_picks()
        fq,am,_ = spettro(tr,*window.value)
        om, fc_fit, rms = fit_brune(fq,am,fmin.value,fmax.value)
        if not (logomega.min <= np.log10(om) <= logomega.max):
            raise ValueError('Plateau del fit fuori dai limiti dei cursori')
        if not (10**fc.min <= fc_fit <= 10**fc.max):
            raise ValueError('fc del fit fuori dai limiti dei cursori')
        state['busy'] = True
        logomega.value = float(np.log10(om))
        fc.value = float(fc_fit)
        state['modo'] = 'automatico'
        state['busy'] = False
        redraw()
        notice.value = f'<b>Fit automatico:</b> RMS log10 = {rms:.3f}. Valuta la qualità fisica del modello!'
    except Exception as exc:
        state['busy'] = False
        notice.value = f'<b style="color:#b91c1c">Fit non eseguito:</b> {html.escape(str(exc))}'


def manual_adjusted(change=None):
    if state['busy']:
        return
    state['modo']='manuale'
    redraw()


def show_records():
    with records:
        clear_output(wait=True)
        if not state['misure']:
            print('Nessuna misura registrata.')
            return
        rows = ['<table style="border-collapse:collapse;width:100%;font-size:13px">'
                '<tr><th>Stazione</th><th>Comp.</th><th>Fase</th><th>Ω₀ (m·s)</th><th>fc (Hz)</th><th>Mw</th><th>Modo</th></tr>']
        for row in state['misure']:
            rows.append('<tr>' + ''.join(f'<td style="padding:4px 8px;border-bottom:1px solid #ddd">{html.escape(str(v))}</td>' for v in
                        (row['stazione'],row['componente'],row['fase'], f"{row['omega0_m_s']:.2e}",
                         f"{row['fc_Hz']:.3f}",f"{row['Mw']:.2f}",row['modo'])) + '</tr>')
        display(HTML(''.join(rows)+'</table>'))


def record_click(_):
    if not state['tracce']:
        return
    try:
        tr = current_trace()
        require_student_picks()
        spettro(tr,*window.value)
        M0,Mw = momento_magnitudo(tr,phase.value,10**logomega.value)
        start,end=window.value
        state['misure'].append(dict(
            stazione=station.value, componente=component.value, fase=phase.value,
            pick_P_s=state['picks'][station.value]['P'],
            pick_S_s=state['picks'][station.value]['S'],
            finestra_inizio_s=start, finestra_fine_s=end, fmin_Hz=fmin.value, fmax_Hz=fmax.value,
            user3_counts_per_m_s2=tr['user3'], omega0_m_s=10**logomega.value,
            fc_Hz=fc.value, M0_Nm=M0, Mw=Mw, Mw_header=tr['mag'], modo=state['modo']))
        show_records()
        # Non espandere automaticamente la tabella: Jupyter Classic non deve comprimere l'output.
        notice.value=f'<b>Misure registrate: {len(state["misure"])}.</b>'
    except Exception as exc:
        notice.value=f'<b style="color:#b91c1c">Non registrata:</b> {html.escape(str(exc))}'


def export_click(_):
    if not state['misure']:
        notice.value='<i>Prima premi Registra misura.</i>'
        return
    output_csv = Path('misure_Brune_studenti.csv')
    with output_csv.open('w', newline='', encoding='utf-8-sig') as fp:
        writer=csv.DictWriter(fp,fieldnames=list(state['misure'][0]))
        writer.writeheader()
        writer.writerows(state['misure'])
    notice.value=f'<b>CSV salvato:</b> {html.escape(str(output_csv.resolve()))}'


def reset_window_click(_):
    configure_window()

load_button.on_click(load_click)
station.observe(station_changed,names='value')
component.observe(component_changed,names='value')
phase.observe(phase_changed,names='value')
window.observe(redraw,names='value')
for control in (fmin,fmax,logomega,fc):
    control.observe(manual_adjusted,names='value')
fit_button.on_click(fit_click)
record_button.on_click(record_click)
export_button.on_click(export_click)
reset_window.on_click(reset_window_click)

# Un comando opzionale per mostrare il codice (utile a chi insegna).
# Per gli studenti, codice nascosto automaticamente dopo Run All.
show_code = widgets.ToggleButton(value=False, description='Mostra codice',
                                 icon='code', tooltip='Mostra/nasconde le celle Python',
                                 layout=widgets.Layout(width='165px'))

def update_code_visibility(change=None):
    show_code.description = 'Nascondi codice' if show_code.value else 'Mostra codice'
    with code_style_output:
        clear_output(wait=True)
        if not show_code.value:
            display(HTML('''<style>
                /* Jupyter Notebook Classic */
                div.code_cell div.input, div.code_cell div.input_prompt {
                    display: none !important;
                }
                /* JupyterLab e Notebook 7 */
                .jp-CodeCell .jp-Cell-inputWrapper { display: none !important; }
                /* Jupyter Classic: no scrolling interno negli output lunghi */
                div.output_scroll { max-height:none !important; height:auto !important;
                                    overflow-y:visible !important; box-shadow:none !important; }
                </style>'''))

show_code.observe(update_code_visibility, names='value')

# Contenuti secondari in sezioni richiudibili: restano fuori dalla vista principale.
questions = widgets.HTML(value='''
<ol style="padding-left:24px;line-height:1.7">
<li>La stima di Mw cambia al variare di stazione, componente o durata della finestra?</li>
<li>Se la banda di fit non comprende fc, quanto sono ben vincolati i parametri?</li>
<li>Quale effetto ha la risoluzione Δf = 1/T per finestre più lunghe?</li>
<li>Il plateau è riconoscibile al di sopra del rumore? Quali effetti possono alterarlo?</li>
<li>Perché Mw dipende dal plateau e non direttamente da fc?</li>
</ol>
<small>Il modello assume propagazione geometrica semplificata, velocità e densità fissate,
coefficiente di radiazione medio e fattore di superficie libera. Non corregge Q e sito.
Per Mw elevati la corner frequency può essere molto bassa e difficile da risolvere.</small>
''')
questions_accordion = widgets.Accordion(children=[questions])
questions_accordion.set_title(0, 'Domande e limiti del modello (apri per leggere)')
questions_accordion.selected_index = None

record_accordion = widgets.Accordion(children=[records])
record_accordion.set_title(0, 'Tabella delle misure registrate')
record_accordion.selected_index = None

# Il resto delle celle lavora dietro le quinte: solo questo pannello è per gli studenti.
panel = widgets.VBox([
    widgets.HBox([
        widgets.HTML(value='<h2 style="margin:2px 0">Esercizio — Calcolo della magnitudo</h2>'),
        show_code
    ], layout=widgets.Layout(justify_content='space-between', align_items='center')),
    widgets.HTML(value='<span style="color:#64748b">Dal sismogramma allo spettro di spostamento e alla magnitudo momento · Mw di catalogo: 6.8</span>'),
    widgets.HTML(value='<h3 style="margin-bottom:4px">0 · Riconosci e marca gli arrivi P e S</h3>'),
    widgets.HBox([path_box, load_button]),
    load_status,
    widgets.HBox([pick_station], layout=widgets.Layout(flex_flow='row wrap')),
    widgets.HTML(value='<div style="color:#526475;font-size:13px">'
                 'Le tre componenti <b>Z (HNZ), N (HNN), E (HNE)</b> hanno '
                 'una scala temporale comune. Usa <b>Zoom +</b> e <b>Zoom −</b> per ingrandire '
                 'o allargare il tempo attorno al cursore rosso; <b>Vista iniziale</b> '
                 'ripristina la finestra originale. Puoi anche trascinare lo slider Zoom temporale. '
                 'Premi <b>Segna P</b> o <b>Segna S</b> per registrare i pick. '
                 '<b>Per i movimenti fini:</b> usa <b>◀ 1 campione</b> e '
                 '<b>1 campione ▶</b> (oppure le frecce dello slider, se il browser '
                 'le supporta). Il tempo preciso è modificabile anche numericamente.</div>'),
    # Controlli di zoom, cursore e pick sopra alle tre forme d'onda.
    pick_view,
    widgets.HBox([pick_zoom_in_button, pick_zoom_out_button, pick_home_button],
                  layout=widgets.Layout(flex_flow='row wrap')),
    pick_time_slider,
    widgets.HBox([pick_left_button, pick_right_button, pick_time_input,
                  pick_P_button, pick_S_button],
                 layout=widgets.Layout(flex_flow='row wrap')),
    pick_graph,
    widgets.HBox([pick_clear_button,pick_export_button],
                 layout=widgets.Layout(flex_flow='row wrap')),
    pick_notice, pick_summary,
    widgets.HTML(value='<h3 style="margin-bottom:4px">1 · Scegli stazione, componente, fase e finestra per lo spettro</h3>'),
    widgets.HBox([station, component, phase], layout=widgets.Layout(flex_flow='row wrap')),
    waveform,
    window,
    widgets.HBox([reset_window, zoom_window]),
    window_label,
    widgets.HTML(value='<h3 style="margin-bottom:4px">2 · Analizza lo spettro</h3>'),
    spectrum_output,
    widgets.HBox([fmin, fmax]),
    logomega, fc,
    widgets.HBox([fit_button, record_button, export_button], layout=widgets.Layout(flex_flow='row wrap')),
    widgets.HTML(value='<h3 style="margin-bottom:4px">3 · Risultati della stima</h3>'),
    metrics, notice,
    record_accordion,
    questions_accordion
], layout=widgets.Layout(width='100%', gap='5px'))

display(panel)
zoom_window.observe(redraw, names='value')
if PICKER_BACKEND != 'widget':
    pick_notice.value = '<i>Modalità compatibile JupyterLite: usa Zoom +/−, Vista iniziale e i pulsanti di spostamento di 1 campione. Non serve la toolbar Matplotlib.</i>'
else:
    pick_notice.value = '<i>Modalità interattiva attiva: clic sul grafico per spostare la linea rossa, frecce ←/→ per rifinirla; Segna P/S per salvarla.</i>'

update_code_visibility()

# Apertura automatica dei dati se la cartella è accanto al notebook.
def initial_exists():
    p = Path(path_box.value)
    return p.is_dir() or (p.is_file() and p.suffix.lower() == '.zip')

if initial_exists():
    load_click()
else:
    load_status.value = '<i>Indica la cartella contenente i file SAC e premi Carica dati.</i>'


In [ ]:
# ---- 5 · ANALISI STATISTICA DELLE MISURE (aggiunta in coda all'interfaccia originale) ----
# Questa cella non modifica i controlli precedenti. Si aggiorna dopo "Registra misura".

stats_group = widgets.Dropdown(
    options=[('Per stazione', 'stazione'),
             ('Per componente / canale', 'componente'),
             ('Per fase P o S', 'fase'),
             ('Per stazione e componente', 'stazione_componente'),
             ('Tutte le misure', 'tutte')],
    value='stazione', description='Raggruppa:',
    style={'description_width': '105px'},
    layout=widgets.Layout(width='320px'))
stats_phase = widgets.Dropdown(
    options=[('Solo fase S', 'S'), ('Solo fase P', 'P'), ('Entrambe (P + S)', 'tutte')],
    value='S', description='Fase inclusa:',
    style={'description_width': '105px'},
    layout=widgets.Layout(width='285px'))
stats_error = widgets.ToggleButtons(
    options=[('Dev. standard (SD)', 'SD'), ('Errore media (SEM)', 'SEM')],
    value='SD', description='Barre d’errore:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='455px'))
stats_unique = widgets.Checkbox(
    value=True,
    description='Una misura per stazione / componente / fase (l’ultima registrata)',
    indent=False,
    layout=widgets.Layout(width='510px'))
stats_remove = widgets.Button(description='Annulla ultima misura', icon='undo')
stats_export = widgets.Button(description='Esporta statistiche CSV', icon='download')
stats_chart = widgets.Output()
stats_table = widgets.Output()
stats_summary = widgets.HTML(value='')
stats_notice = widgets.HTML(value='')


def statistiche_misure_selezionate():
    """Le misure vengono lette direttamente dalla tabella registrata nell'interfaccia."""
    selected = [r for r in state['misure']
                if stats_phase.value == 'tutte' or r['fase'] == stats_phase.value]
    if stats_unique.value:
        last = {}
        for r in selected:
            last[(r['stazione'], r['componente'], r['fase'])] = r
        selected = list(last.values())
    return selected


def statistiche_gruppo(r):
    if stats_group.value == 'tutte':
        return 'Tutte'
    if stats_group.value == 'stazione_componente':
        return f"{r['stazione']} · {r['componente']}"
    return str(r[stats_group.value])


def magnitudo_catalogo(records):
    """Magnitudo di riferimento coerente con le registrazioni selezionate."""
    vals = [float(r['Mw_header']) for r in records
            if r.get('Mw_header') is not None and np.isfinite(r['Mw_header'])]
    return float(np.median(vals)) if vals else None


def calcola_statistiche_gruppi(records=None):
    """n, media, SD, SEM, bias rispetto al catalogo per i gruppi scelti."""
    if records is None:
        records = statistiche_misure_selezionate()
    groups = {}
    for r in records:
        groups.setdefault(statistiche_gruppo(r), []).append(r)
    ref_mag = magnitudo_catalogo(records)
    result = []
    for group, items in sorted(groups.items()):
        vals = np.asarray([float(r['Mw']) for r in items], dtype=float)
        n = len(vals)
        sd = float(np.std(vals, ddof=1)) if n >= 2 else np.nan
        sem = sd / np.sqrt(n) if n >= 2 else np.nan
        mean = float(np.mean(vals))
        result.append({
            'gruppo': group, 'n_misure': n,
            'n_stazioni': len({r['stazione'] for r in items}),
            'Mw_media': mean, 'Mw_SD': sd, 'Mw_SEM': sem,
            'Mw_catalogo': ref_mag,
            'bias_rispetto_catalogo': mean - ref_mag if ref_mag is not None else np.nan,
        })
    return result


def statistiche_stazioni_equipesate(records):
    """Prima si fa la media entro ogni stazione; poi tra le stazioni."""
    by_station = {}
    for r in records:
        by_station.setdefault(r['stazione'], []).append(float(r['Mw']))
    means = np.asarray([np.mean(v) for v in by_station.values()], dtype=float)
    if len(means) == 0:
        return None
    sd = float(np.std(means, ddof=1)) if len(means) >= 2 else np.nan
    return dict(n_stazioni=len(means), Mw_media=float(np.mean(means)),
                Mw_SD=sd, Mw_SEM=sd / np.sqrt(len(means)) if len(means) >= 2 else np.nan)


def aggiorna_statistiche(_=None):
    selected = statistiche_misure_selezionate()
    groups = calcola_statistiche_gruppi(selected)
    ref_mag = magnitudo_catalogo(selected)

    with stats_chart:
        clear_output(wait=True)
        if not selected:
            display(HTML('<i>Registra almeno una misura della fase selezionata per visualizzare il confronto.</i>'))
        else:
            fig, ax = plt.subplots(figsize=(max(9.5, min(15, 0.85 * len(groups) + 5)), 4.5),
                                   constrained_layout=True)
            xx = np.arange(len(groups))
            for j, group in enumerate(groups):
                y = np.asarray([float(r['Mw']) for r in selected
                                if statistiche_gruppo(r) == group['gruppo']])
                offset = np.linspace(-0.14, 0.14, len(y)) if len(y) > 1 else np.array([0.])
                ax.scatter(j + offset, y, color='#728399', s=35, alpha=.70, zorder=3,
                           label='Misure individuali' if j == 0 else None)
            means = np.asarray([r['Mw_media'] for r in groups], dtype=float)
            err_field = 'Mw_SD' if stats_error.value == 'SD' else 'Mw_SEM'
            errors = np.asarray([r[err_field] for r in groups], dtype=float)
            ax.errorbar(xx, means, yerr=np.where(np.isfinite(errors), errors, 0.0),
                        fmt='D', color='#0c8d84', ecolor='#0c8d84', capsize=5,
                        markersize=7, linewidth=1.8, zorder=5,
                        label='Media ± ' + stats_error.value + ' (n ≥ 2)')
            if ref_mag is not None:
                ax.axhline(ref_mag, color='#b25050', linestyle='--', linewidth=1.5,
                           label=f'Mw catalogo = {ref_mag:.2f}')
            ax.set_xticks(xx)
            ax.set_xticklabels([f"{r['gruppo']}\n(n={r['n_misure']})" for r in groups],
                               rotation=25 if len(groups) > 4 else 0,
                               ha='right' if len(groups) > 4 else 'center')
            ax.set(ylabel='Magnitudo momento Mw',
                   title='Distribuzione delle stime per gruppo')
            ax.set_xlim(-.55, len(groups) - .45)
            ax.legend(loc='best', fontsize=9)
            mostra_figura_png(fig)

    with stats_table:
        clear_output(wait=True)
        if groups:
            columns = [('Gruppo', 'gruppo'), ('Misure', 'n_misure'),
                       ('Stazioni', 'n_stazioni'), ('Mw media', 'Mw_media'),
                       ('SD', 'Mw_SD'), ('SEM', 'Mw_SEM'),
                       ('Scarto da catalogo', 'bias_rispetto_catalogo')]
            table = ['<table style="border-collapse:collapse;width:100%;font-size:13px">',
                     '<thead><tr>' + ''.join(
                         '<th style="border-bottom:1px solid #aaa;padding:6px 8px;text-align:left">'
                         + html.escape(label) + '</th>' for label, _ in columns) + '</tr></thead><tbody>']
            for r in groups:
                pieces = []
                for _, field in columns:
                    v = r[field]
                    if isinstance(v, (float, np.floating)):
                        shown = f'{v:+.3f}' if field == 'bias_rispetto_catalogo' and np.isfinite(v) else (
                            f'{v:.3f}' if np.isfinite(v) else '—')
                    else:
                        shown = str(v)
                    pieces.append('<td style="padding:5px 8px;border-bottom:1px solid #e0e0e0">'
                                  + html.escape(shown) + '</td>')
                table.append('<tr>' + ''.join(pieces) + '</tr>')
            display(HTML(''.join(table) + '</tbody></table>'))

    combined = statistiche_stazioni_equipesate(selected)
    if combined is None:
        stats_summary.value = '<span style="color:#64748b">Le statistiche si aggiorneranno dopo ogni misura registrata.</span>'
        return
    mean = combined['Mw_media']
    unc = combined['Mw_SD' if stats_error.value == 'SD' else 'Mw_SEM']
    uncertainty = (f' ± {unc:.3f}' if np.isfinite(unc)
                   else ' (incertezza non stimabile: serve almeno una seconda stazione)')
    bias = (f'{mean - ref_mag:+.3f}' if ref_mag is not None else 'n.d.')
    mixed = ('<br><b>Attenzione:</b> P e S sono combinate; i loro errori sistematici '
             'possono essere diversi.' if stats_phase.value == 'tutte' else '')
    stats_summary.value = (
        '<div style="padding:12px;border:1px solid #cfd9e0;border-radius:8px">'
        f'<b>Stima combinata (una media per stazione):</b> Mw = '
        f'<b>{mean:.3f}{uncertainty}</b> &nbsp;·&nbsp; '
        f'{combined["n_stazioni"]} stazioni &nbsp;·&nbsp; '
        f'scarto dal catalogo: {bias}<br>'
        '<small>Ogni stazione ha lo stesso peso, anche quando sono presenti più componenti. '
        'SD rappresenta la dispersione tra le medie delle stazioni; SEM = SD/√N è l’errore '
        'standard della loro media solo sotto ipotesi di indipendenza. '
        'Non sono incluse le incertezze sistematiche dovute a modello, attenuazione, '
        'sito e calibrazione. Con un solo dato in un gruppo, SD e SEM non sono stimabili.'
        + mixed + '</small></div>')


def annulla_ultima_misura(_):
    if not state['misure']:
        stats_notice.value = '<i>Non ci sono misure da annullare.</i>'
        return
    state['misure'].pop()
    show_records()  # Mantiene sincronizzata anche la tabella originale.
    aggiorna_statistiche()
    stats_notice.value = '<span style="color:#166534">Ultima misura annullata.</span>'


def esporta_statistiche(_):
    rows = calcola_statistiche_gruppi()
    if not rows:
        stats_notice.value = '<i>Registra almeno una misura prima di esportare.</i>'
        return
    target = Path('riepilogo_statistico_Brune.csv')
    try:
        with target.open('w', newline='', encoding='utf-8-sig') as fp:
            writer = csv.DictWriter(fp, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        stats_notice.value = f'<b>Riepilogo salvato:</b> {html.escape(str(target.resolve()))}'
    except Exception as exc:
        stats_notice.value = f'<b style="color:#b91c1c">Esportazione fallita:</b> {html.escape(str(exc))}'


# Il pulsante originale registra già le misure; il secondo callback aggiorna i grafici.
record_button.on_click(aggiorna_statistiche)
for control in (stats_group, stats_phase, stats_error, stats_unique):
    control.observe(aggiorna_statistiche, names='value')
stats_remove.on_click(annulla_ultima_misura)
stats_export.on_click(esporta_statistiche)

# Aggiunge la sezione 5 *in fondo allo stesso pannello originale*.
# Non vengono cambiati menu, grafici, finestre o pulsanti delle sezioni 1–4.
panel.children = panel.children + (
    widgets.HTML(value='<h3 style="margin:15px 0 4px">4 · Confronta le misure e valuta l’incertezza</h3>'),
    widgets.HTML(value='<span style="color:#64748b">Registra misure da stazioni e componenti diverse. '
                 'Le medie e gli errori si aggiornano automaticamente.</span>'),
    widgets.HBox([stats_group, stats_phase], layout=widgets.Layout(flex_flow='row wrap')),
    widgets.HBox([stats_error, stats_unique], layout=widgets.Layout(flex_flow='row wrap')),
    stats_chart,
    stats_summary,
    stats_table,
    widgets.HBox([stats_remove, stats_export], layout=widgets.Layout(flex_flow='row wrap')),
    stats_notice,
    widgets.HTML(value='<small style="color:#64748b">Per studiare la sensibilità alla finestra, '
                 'disattiva “Una misura per stazione/componente/fase”, così includerai anche '
                 'le prove ripetute. Per una stima finale è preferibile non trattare '
                 'prove correlate sulla stessa traccia come osservazioni indipendenti.</small>'),
)
aggiorna_statistiche()
